# MILESTONE 2

#Task 7:  KPI Engineering + Feature Selection

We define the three KPIs we'll compute, what each one measures and the formula used.

**KPI 1: Assets per Capita**

**Purpose:** Measure military asset availability relative to manpower.

**Formula:** (total_aircraft + tanks + total_naval_assets)/active_personnel



**KPI 2: Budget-to-GDP Ratio**

**Purpose:** Measure defense spending intensity relative to economy size.

**Formula:** defense_budget_usd/gdp_usd

**KPI 3: Power Index Rank Gap**

**Purpose:** Measure how far a country is from the top-ranked nation.

**Formula:** rank(country)-rank(top_country)

Assets per Capita measures how mechanised a military is.

Budget-to-GDP Ratio measures spending priority.

Rank Gap measures how far behind the top-ranked country each nation is.

We load the clean dataset and calculate how many total hardware assets each country has per active soldier.


In [ ]:
# Load

import pandas as pd
import numpy as np

df = pd.read_csv("military_selected_clean.csv")

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(df.head())

In [ ]:
#KPI 1 — Assets per Capita
# (tanks + total_aircraft + total_naval_assets) / active_personnel
# Tells us how many hardware assets exist per soldier

df["assets_per_capita"] = np.where(
    df["active_personnel"] > 0,
    (df["tanks"] + df["total_aircraft"] + df["total_naval_assets"]) / df["active_personnel"],
    np.nan
).round(4)

print("KPI 1 — Assets per Capita")
print(df[["country", "assets_per_capita"]].sort_values("assets_per_capita", ascending=False).head(10).to_string(index=False))


np.where() is used to avoid dividing by zero ie. if a country has 0 soldiers it gets NaN instead of a crash. Higher values mean a more mechanised military relative to its size. Iceland gets NaN since it has no active personnel

We calculate what percentage of each country's economy is spent on its military.

In [ ]:
# KPI 2 — Budget-to-GDP Ratio (%)
# (defense_budget_usd / gdp_usd)
# Tells us what % of the economy goes to military

df["budget_to_gdp_ratio"] = np.where(
    df["gdp_usd"] > 0,
    (df["defense_budget_usd"] / df["gdp_usd"]),
    np.nan
).round(4)

print("\nKPI 2 — Budget to GDP ")
print(df[["country", "defense_budget_usd", "gdp_usd", "budget_to_gdp_ratio"]].sort_values("budget_to_gdp_ratio", ascending=False).head(10).to_string(index=False))



This levels the playing field between rich and poor countries. A small country spending 8% of its GDP is prioritising military more than a large country spending 3%, even if the large country's raw budget is 100x bigger.

We calculate how many rank positions each country is behind the #1 ranked nation.

In [ ]:
# KPI 3 — Power Index Rank Gap
# country_rank - 1  (since rank 1 = top country)
# Tells us how far each country is from the #1 spot

top_country = df.loc[df["power_index_rank"] == 1, "country"].values[0]

df["rank_gap"] = (df["power_index_rank"] - 1).astype(int)

print(f"\nKPI 3 — Rank Gap from {top_country} (Rank 1)")
print(df[["country", "power_index_rank", "rank_gap"]].sort_values("rank_gap").head(10).to_string(index=False))


 rank - 1 gives a gap of 0 for the top country and 144 for the last. This makes rank comparisons easier to read in charts than raw rank numbers.

We preview the full KPI table sorted by rank and save the final file.

In [ ]:
# CELL 5: View all 3 KPIs together
print("\nFull KPI Table (sorted by rank):")
print(
    df[["country", "power_index_rank", "assets_per_capita", "budget_to_gdp_ratio", "rank_gap"]]
    .sort_values("power_index_rank")
    .to_string(index=False)
)


In [ ]:
# CELL 6: Save & Download
df.to_csv("military_kpi.csv", index=False)
print(f"\nSaved: military_kpi.csv ({df.shape[0]} rows x {df.shape[1]} cols)")

from google.colab import files
files.download("military_kpi.csv")


Output is military_kpi.csv with all original columns plus the 3 new KPI columns.

# KPI Validation

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("military_kpi.csv")
print(f"{df.shape[0]} rows x {df.shape[1]} columns")

We check that no KPI calculation produced an invalid result due to division by zero.

In [ ]:
# Check for Division by Zero
# assets_per_capita divides by active_personnel
# budget_to_gdp_pct divides by gdp_usd

print("\nCHECK 1 — Division by Zero")

zero_personnel = df[df["active_personnel"] == 0]["country"].tolist()
print(f"Countries with 0 active_personnel: {zero_personnel}")
print(f"assets_per_capita NaN for these: {df.loc[df['active_personnel'] == 0, 'assets_per_capita'].isna().all()}")
print(f"Any infinite values: {df[['assets_per_capita','budget_to_gdp_ratio','rank_gap']].isin([np.inf,-np.inf]).any().any()}")


Iceland (0 active personnel) correctly shows NaN for assets_per_capita. isin([np.inf, -np.inf]) checks for infinite values which would indicate a divide-by-zero that wasn't caught.

We check that all KPI values fall within sensible boundaries for each metric.

In [ ]:
# Check Realistic Ranges
print("\nCHECK 2 — Realistic Ranges")

for col, low, high in [
    ("assets_per_capita",  0, 1.0),
    ("budget_to_gdp_ratio",  0, 15.0),
    ("rank_gap",           0, 144),
]:
    col_data = df[col].dropna()
    out = df[(df[col] < low) | (df[col] > high)]["country"].tolist()
    print(f"\n{col}")
    print(f"  Min: {col_data.min():.4f} ({df.loc[col_data.idxmin(),'country']})")
    print(f"  Max: {col_data.max():.4f} ({df.loc[col_data.idxmax(),'country']})")
    print(f"  Out of range [{low} - {high}]: {out if out else 'None'}")


assets_per_capita should be below 1.0

 budget_to_gdp_ratio below 15

 rank_gap between 0 and 144.

 Anything outside these ranges is flagged as outlier.

We manually recalculate all 3 KPIs for 5 known countries and compare to what's stored.

In [ ]:
# Manual Spot Check
# Recalculate KPIs by hand for 5 countries and compare

print("\nCHECK 3 — Manual Spot Check")

for country in ["United States", "China", "India", "Russia", "Iceland"]:
    r = df[df["country"] == country].iloc[0]
    print(f"\n{country}")

    # assets_per_capita
    if r["active_personnel"] == 0:
        print(f"  assets_per_capita : expected NaN → {r['assets_per_capita']} {'✅' if pd.isna(r['assets_per_capita']) else '❌'}")
    else:
        exp = round((r["tanks"] + r["total_aircraft"] + r["total_naval_assets"]) / r["active_personnel"], 4)
        print(f"  assets_per_capita : expected {exp} → {r['assets_per_capita']} {'✅' if exp == r['assets_per_capita'] else '❌'}")

    # budget_to_gdp_pct
    exp = round(r["defense_budget_usd"] / r["gdp_usd"] * 100, 4)
    print(f"  budget_to_gdp_ratio : expected {exp} → {r['budget_to_gdp_ratio']} {'✅' if exp == r['budget_to_gdp_ratio'] else '❌'}")

    # rank_gap
    exp = int(r["power_index_rank"] - 1)
    print(f"  rank_gap          : expected {exp} → {r['rank_gap']} {'✅' if exp == r['rank_gap'] else '❌'}")

If the manually computed value matches the stored value to 4 decimal places it prints ✅, otherwise ❌. This confirms our formulas are working correctly end to end. Iceland is included deliberately to verify the NaN edge case.

# Metadata Enrichment

In [ ]:
import pandas as pd

df = pd.read_csv("military_kpi.csv")
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")

We add three descriptive columns to the dataset: region, continent and alliance — by mapping each country to its values from a lookup dictionary.

In [ ]:
# ── CELL 2: Define metadata for all 145 countries ────────────────
# Format: "Country": ("Region", "Continent", "Alliance")
# Alliance values: "NATO", "CSTO", "None"
# CSTO = Collective Security Treaty Organisation (Russia-led bloc)

METADATA = {
    "Afghanistan":                    ("South Asia",         "Asia",          "None"),
    "Albania":                        ("Southern Europe",    "Europe",        "NATO"),
    "Algeria":                        ("North Africa",       "Africa",        "None"),
    "Angola":                         ("Central Africa",     "Africa",        "None"),
    "Argentina":                      ("South America",      "Americas",      "None"),
    "Armenia":                        ("Central Asia",       "Asia",          "CSTO"),
    "Australia":                      ("Oceania",            "Oceania",       "None"),
    "Austria":                        ("Western Europe",     "Europe",        "None"),
    "Azerbaijan":                     ("Central Asia",       "Asia",          "None"),
    "Bahrain":                        ("Middle East",        "Asia",          "None"),
    "Bangladesh":                     ("South Asia",         "Asia",          "None"),
    "Belarus":                        ("Eastern Europe",     "Europe",        "CSTO"),
    "Belgium":                        ("Western Europe",     "Europe",        "NATO"),
    "Belize":                         ("Central America",    "Americas",      "None"),
    "Benin":                          ("West Africa",        "Africa",        "None"),
    "Bhutan":                         ("South Asia",         "Asia",          "None"),
    "Bolivia":                        ("South America",      "Americas",      "None"),
    "Bosnia and Herzegovina":         ("Southern Europe",    "Europe",        "None"),
    "Botswana":                       ("Southern Africa",    "Africa",        "None"),
    "Brazil":                         ("South America",      "Americas",      "None"),
    "Bulgaria":                       ("Eastern Europe",     "Europe",        "NATO"),
    "Burkina Faso":                   ("West Africa",        "Africa",        "None"),
    "Cambodia":                       ("Southeast Asia",     "Asia",          "None"),
    "Cameroon":                       ("Central Africa",     "Africa",        "None"),
    "Canada":                         ("North America",      "Americas",      "NATO"),
    "Central African Republic":       ("Central Africa",     "Africa",        "None"),
    "Chad":                           ("Central Africa",     "Africa",        "None"),
    "Chile":                          ("South America",      "Americas",      "None"),
    "China":                          ("East Asia",          "Asia",          "None"),
    "Colombia":                       ("South America",      "Americas",      "None"),
    "Croatia":                        ("Southern Europe",    "Europe",        "NATO"),
    "Cuba":                           ("Caribbean",          "Americas",      "None"),
    "Czechia":                        ("Eastern Europe",     "Europe",        "NATO"),
    "Democratic Republic of the Congo": ("Central Africa",  "Africa",        "None"),
    "Denmark":                        ("Northern Europe",    "Europe",        "NATO"),
    "Dominican Republic":             ("Caribbean",          "Americas",      "None"),
    "Ecuador":                        ("South America",      "Americas",      "None"),
    "Egypt":                          ("North Africa",       "Africa",        "None"),
    "El Salvador":                    ("Central America",    "Americas",      "None"),
    "Eritrea":                        ("East Africa",        "Africa",        "None"),
    "Estonia":                        ("Northern Europe",    "Europe",        "NATO"),
    "Ethiopia":                       ("East Africa",        "Africa",        "None"),
    "Finland":                        ("Northern Europe",    "Europe",        "NATO"),
    "France":                         ("Western Europe",     "Europe",        "NATO"),
    "Gabon":                          ("Central Africa",     "Africa",        "None"),
    "Georgia":                        ("Central Asia",       "Asia",          "None"),
    "Germany":                        ("Western Europe",     "Europe",        "NATO"),
    "Ghana":                          ("West Africa",        "Africa",        "None"),
    "Greece":                         ("Southern Europe",    "Europe",        "NATO"),
    "Guatemala":                      ("Central America",    "Americas",      "None"),
    "Honduras":                       ("Central America",    "Americas",      "None"),
    "Hungary":                        ("Eastern Europe",     "Europe",        "NATO"),
    "Iceland":                        ("Northern Europe",    "Europe",        "NATO"),
    "India":                          ("South Asia",         "Asia",          "None"),
    "Indonesia":                      ("Southeast Asia",     "Asia",          "None"),
    "Iran":                           ("Middle East",        "Asia",          "None"),
    "Iraq":                           ("Middle East",        "Asia",          "None"),
    "Ireland":                        ("Western Europe",     "Europe",        "None"),
    "Israel":                         ("Middle East",        "Asia",          "None"),
    "Italy":                          ("Southern Europe",    "Europe",        "NATO"),
    "Ivory Coast":                    ("West Africa",        "Africa",        "None"),
    "Japan":                          ("East Asia",          "Asia",          "None"),
    "Jordan":                         ("Middle East",        "Asia",          "None"),
    "Kazakhstan":                     ("Central Asia",       "Asia",          "CSTO"),
    "Kenya":                          ("East Africa",        "Africa",        "None"),
    "Kosovo":                         ("Southern Europe",    "Europe",        "None"),
    "Kuwait":                         ("Middle East",        "Asia",          "None"),
    "Kyrgyzstan":                     ("Central Asia",       "Asia",          "CSTO"),
    "Laos":                           ("Southeast Asia",     "Asia",          "None"),
    "Latvia":                         ("Northern Europe",    "Europe",        "NATO"),
    "Lebanon":                        ("Middle East",        "Asia",          "None"),
    "Liberia":                        ("West Africa",        "Africa",        "None"),
    "Libya":                          ("North Africa",       "Africa",        "None"),
    "Lithuania":                      ("Northern Europe",    "Europe",        "NATO"),
    "Luxembourg":                     ("Western Europe",     "Europe",        "NATO"),
    "Madagascar":                     ("East Africa",        "Africa",        "None"),
    "Malaysia":                       ("Southeast Asia",     "Asia",          "None"),
    "Mali":                           ("West Africa",        "Africa",        "None"),
    "Mauritania":                     ("West Africa",        "Africa",        "None"),
    "Mexico":                         ("North America",      "Americas",      "None"),
    "Moldova":                        ("Eastern Europe",     "Europe",        "None"),
    "Mongolia":                       ("East Asia",          "Asia",          "None"),
    "Montenegro":                     ("Southern Europe",    "Europe",        "NATO"),
    "Morocco":                        ("North Africa",       "Africa",        "None"),
    "Mozambique":                     ("East Africa",        "Africa",        "None"),
    "Myanmar":                        ("Southeast Asia",     "Asia",          "None"),
    "Namibia":                        ("Southern Africa",    "Africa",        "None"),
    "Nepal":                          ("South Asia",         "Asia",          "None"),
    "Netherlands":                    ("Western Europe",     "Europe",        "NATO"),
    "New Zealand":                    ("Oceania",            "Oceania",       "None"),
    "Nicaragua":                      ("Central America",    "Americas",      "None"),
    "Niger":                          ("West Africa",        "Africa",        "None"),
    "Nigeria":                        ("West Africa",        "Africa",        "None"),
    "North Korea":                    ("East Asia",          "Asia",          "None"),
    "North Macedonia":                ("Southern Europe",    "Europe",        "NATO"),
    "Norway":                         ("Northern Europe",    "Europe",        "NATO"),
    "Oman":                           ("Middle East",        "Asia",          "None"),
    "Pakistan":                       ("South Asia",         "Asia",          "None"),
    "Panama":                         ("Central America",    "Americas",      "None"),
    "Paraguay":                       ("South America",      "Americas",      "None"),
    "Peru":                           ("South America",      "Americas",      "None"),
    "Philippines":                    ("Southeast Asia",     "Asia",          "None"),
    "Poland":                         ("Eastern Europe",     "Europe",        "NATO"),
    "Portugal":                       ("Southern Europe",    "Europe",        "NATO"),
    "Qatar":                          ("Middle East",        "Asia",          "None"),
    "Republic of the Congo":          ("Central Africa",     "Africa",        "None"),
    "Romania":                        ("Eastern Europe",     "Europe",        "NATO"),
    "Russia":                         ("Eastern Europe",     "Europe",        "None"),
    "Saudi Arabia":                   ("Middle East",        "Asia",          "None"),
    "Senegal":                        ("West Africa",        "Africa",        "None"),
    "Serbia":                         ("Southern Europe",    "Europe",        "None"),
    "Sierra Leone":                   ("West Africa",        "Africa",        "None"),
    "Singapore":                      ("Southeast Asia",     "Asia",          "None"),
    "Slovakia":                       ("Eastern Europe",     "Europe",        "NATO"),
    "Slovenia":                       ("Southern Europe",    "Europe",        "NATO"),
    "Somalia":                        ("East Africa",        "Africa",        "None"),
    "South Africa":                   ("Southern Africa",    "Africa",        "None"),
    "South Korea":                    ("East Asia",          "Asia",          "None"),
    "South Sudan":                    ("East Africa",        "Africa",        "None"),
    "Spain":                          ("Southern Europe",    "Europe",        "NATO"),
    "Sri Lanka":                      ("South Asia",         "Asia",          "None"),
    "Sudan":                          ("North Africa",       "Africa",        "None"),
    "Suriname":                       ("South America",      "Americas",      "None"),
    "Sweden":                         ("Northern Europe",    "Europe",        "NATO"),
    "Switzerland":                    ("Western Europe",     "Europe",        "None"),
    "Syria":                          ("Middle East",        "Asia",          "None"),
    "Taiwan":                         ("East Asia",          "Asia",          "None"),
    "Tajikistan":                     ("Central Asia",       "Asia",          "CSTO"),
    "Tanzania":                       ("East Africa",        "Africa",        "None"),
    "Thailand":                       ("Southeast Asia",     "Asia",          "None"),
    "Tunisia":                        ("North Africa",       "Africa",        "None"),
    "Turkey":                         ("Southern Europe",    "Europe",        "NATO"),
    "Turkmenistan":                   ("Central Asia",       "Asia",          "None"),
    "Uganda":                         ("East Africa",        "Africa",        "None"),
    "Ukraine":                        ("Eastern Europe",     "Europe",        "None"),
    "United Arab Emirates":           ("Middle East",        "Asia",          "None"),
    "United Kingdom":                 ("Western Europe",     "Europe",        "NATO"),
    "United States":                  ("North America",      "Americas",      "NATO"),
    "Uruguay":                        ("South America",      "Americas",      "None"),
    "Uzbekistan":                     ("Central Asia",       "Asia",          "CSTO"),
    "Venezuela":                      ("South America",      "Americas",      "None"),
    "Vietnam":                        ("Southeast Asia",     "Asia",          "None"),
    "Yemen":                          ("Middle East",        "Asia",          "None"),
    "Zambia":                         ("Southern Africa",    "Africa",        "None"),
    "Zimbabwe":                       ("Southern Africa",    "Africa",        "None"),
}


In [ ]:
# Add the 3 new columns
df["region"]        = df["country"].map(lambda c: METADATA.get(c, ("Unknown", "Unknown", "None"))[0])
df["continent"]     = df["country"].map(lambda c: METADATA.get(c, ("Unknown", "Unknown", "None"))[1])
df["alliance_flag"] = df["country"].map(lambda c: METADATA.get(c, ("Unknown", "Unknown", "None"))[2])

print("New columns added: region, continent, alliance_flag")
print(df[["country", "region", "continent", "alliance_flag"]].head(15).to_string(index=False))


# Quick check
print(f"\nCountries unmapped (Unknown): {(df['region'] == 'Unknown').sum()}")
print(f"\nAlliance breakdown:")
print(df["alliance_flag"].value_counts().to_string())
print(f"\nContinent breakdown:")
print(df["continent"].value_counts().to_string())


df["country"].map() looks up each country name in the METADATA dictionary and pulls out the corresponding value.

Alliance values are NATO, CSTO or None.

The quick check at the end confirms no countries were left as Unknown.

We save the final enriched dataset with all original columns, KPIs and metadata included.

In [ ]:
# Save & Download
df.to_csv("military_final.csv", index=False)
print(f"\nSaved: military_final.csv ({df.shape[0]} rows x {df.shape[1]} cols)")

from google.colab import files
files.download("military_final.csv")

Output is military_final.csv dataset. It has 145 rows, all 9 original columns, 3 KPI columns and 3 metadata columns.